<a href="https://colab.research.google.com/github/soujoudgahgah/eczema-classification-finetuning/blob/main/eczema_model_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gahgahsoujoud","key":"9fe90d1d75a9f136139ce236a0c4cfbc"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle

In [ ]:
!kaggle datasets download -d shubhamgoel27/dermnet
!unzip -q dermnet.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/shubhamgoel27/dermnet
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 1.72G/1.72G [00:19<00:00, 96.0MB/s]



In [ ]:
import os
for folder in os.listdir('/content/data/train'):
    path = f'/content/data/train/{folder}'
    print(folder, '->', len(os.listdir(path)), 'images')

Seborrheic Keratoses and other Benign Tumors -> 1371 images
Poison Ivy Photos and other Contact Dermatitis -> 260 images
Herpes HPV and other STDs Photos -> 405 images
Nail Fungus and other Nail Disease -> 1040 images
Scabies Lyme Disease and other Infestations and Bites -> 431 images
Bullous Disease Photos -> 448 images
Hair Loss Photos Alopecia and other Hair Diseases -> 239 images
Light Diseases and Disorders of Pigmentation -> 568 images
Tinea Ringworm Candidiasis and other Fungal Infections -> 1300 images
Acne and Rosacea Photos -> 840 images
Exanthems and Drug Eruptions -> 404 images
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions -> 1149 images
Systemic Disease -> 606 images
Eczema Photos -> 1235 images
Urticaria Hives -> 212 images
Atopic Dermatitis Photos -> 489 images
Melanoma Skin Cancer Nevi and Moles -> 463 images
Warts Molluscum and other Viral Infections -> 1086 images
Cellulitis Impetigo and other Bacterial Infections -> 288 images
Vasculitis Photos -

In [ ]:
import shutil, os

base = '/content/data'
target = '/content/dataset'

categories = {
    'eczema': ['Eczema Photos', 'Atopic Dermatitis Photos'],
    'non_eczema': ['Acne and Rosacea Photos', 'Melanoma Skin Cancer Nevi and Moles', 'Warts Molluscum and other Viral Infections', 'Psoriasis pictures Lichen Planus and related diseases']
}

for split in ['train', 'test']:
    for new_class, sources in categories.items():
        dest = f'{target}/{split}/{new_class}'
        os.makedirs(dest, exist_ok=True)
        for src_folder in sources:
            src = f'{base}/{split}/{src_folder}'
            if os.path.exists(src):
                for f in os.listdir(src):
                    s = f'{src}/{f}'
                    d = f'{dest}/{src_folder}_{f}'  # prefix avoids filename collisions
                    if os.path.isfile(s):
                        shutil.copy(s, d)

print("Done organizing")

Done organizing


In [ ]:
for split in ['train', 'test']:
    for cls in ['eczema', 'non_eczema']:
        path = f'/content/dataset/{split}/{cls}'
        print(split, cls, '->', len(os.listdir(path)))

train eczema -> 1724
train non_eczema -> 3794
test eczema -> 432
test non_eczema -> 1052


In [ ]:
import os
print(os.listdir('/content/dataset/train'))

['non_eczema', 'eczema']


In [ ]:
import shutil
shutil.rmtree('/content/dataset/train/healthy', ignore_errors=True)
shutil.rmtree('/content/dataset/test/healthy', ignore_errors=True)

In [ ]:
print(os.listdir('/content/dataset/train'))
print(os.listdir('/content/dataset/test'))


['non_eczema', 'eczema']
['non_eczema', 'eczema']


In [ ]:
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/train',
    image_size=img_size,
    batch_size=batch_size,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/test',
    image_size=img_size,
    batch_size=batch_size,
    label_mode='binary'
)

class_names = train_ds.class_names
print(class_names)

Found 5518 files belonging to 2 classes.
Found 1484 files belonging to 2 classes.
['eczema', 'non_eczema']


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
base_model.trainable = True


for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # معدل تعلم منخفض جداً
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history_fine = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=8
)

In [ ]:
model.save('/content/eczema_model_finetuned.keras')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil
shutil.copy('/content/eczema_model_finetuned.h5', '/content/drive/MyDrive/eczema_model_finetuned.h5')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred.extend((preds.flatten() > 0.5).astype(int))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
history_fine2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

In [ ]:
model.save('/content/eczema_model_finetuned.keras')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil
shutil.copy('/content/eczema_model_finetuned.keras', '/content/drive/MyDrive/eczema_model_finetuned.keras')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred.extend((preds.flatten() > 0.5).astype(int))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 12))
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        pred_label = class_names[int(preds[i][0] > 0.5)]
        true_label = class_names[int(labels[i])]
        confidence = preds[i][0] if preds[i][0] > 0.5 else 1 - preds[i][0]
        color = "green" if pred_label == true_label else "red"
        plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence*100:.0f}%)", color=color)
        plt.axis("off")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
model = tf.keras.models.load_model('/content/drive/MyDrive/eczema_model_finetuned.keras')

Mounted at /content/drive
